# Hyrax Nested Pandas Dataset Example

Minimal example using `NestedPandasDataset` with a nested-pandas parquet file.

In [ ]:
from pathlib import Path
import tempfile

import pandas as pd
import nested_pandas as npd
import hyrax

In [ ]:
tmp_dir = Path(tempfile.mkdtemp())
data_path = tmp_dir / "nested_sources.parquet"

flat = pd.DataFrame(
    {
        "object_id": ["obj_1", "obj_2", "obj_3"],
        "ra": [10.1, 10.2, 10.3],
        "time": [[1.0, 2.0], [3.0, 4.0, 5.0], [6.0]],
        "flux": [[11.0, 12.0], [21.0, 22.0, 23.0], [31.0]],
    }
)
table = npd.NestedFrame.from_lists(
    flat,
    base_columns=["object_id", "ra"],
    list_columns=["time", "flux"],
    name="lightcurve",
)
table.to_parquet(data_path)
data_path

In [ ]:
h = hyrax.Hyrax()
h.config["data_request"] = {
    "train": {
        "data": {
            "dataset_class": "NestedPandasDataset",
            "data_location": str(data_path),
            "fields": ["object_id", "ra", "lightcurve.flux"],
            "primary_id_field": "object_id",
        }
    }
}
dataset = h.prepare()["train"]
len(dataset)

In [ ]:
sample = dataset[0]
sample